<a href="https://colab.research.google.com/github/edwardsnj/glygen-colab-notebooks/blob/main/variants/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
REPO_BASE = "https://raw.githubusercontent.com/edwardsnj/glygen-colab-notebooks/refs/heads/main"
NOTEBOOK = "variants"

import httpimport
with httpimport.remote_repo(REPO_BASE + "/" + NOTEBOOK):
  import make_plotdata, map_datasets, run_binomial_test

!rm -rf data reviewed plots
!mkdir -p data reviewed plots

GLYGEN_DATA_REVIEWED = "https://data.glygen.org/ln2data/releases/data/current/reviewed"

human_protein_mutation_files = """
  human_protein_mutation_cancer_all.csv
  human_protein_mutation_germline_all.csv
""".split()

human_proteoform_glycosylation_site_files = """
human_proteoform_glycosylation_sites_carbbank.csv
human_proteoform_glycosylation_sites_c_man.csv
human_proteoform_glycosylation_sites_diabetes_glycomic.csv
human_proteoform_glycosylation_sites_embl.csv
human_proteoform_glycosylation_sites_glycomeatlas.csv
human_proteoform_glycosylation_sites_glyconnect.csv
human_proteoform_glycosylation_sites_gptwiki.csv
human_proteoform_glycosylation_sites_harvard.csv
human_proteoform_glycosylation_sites_literature.csv
human_proteoform_glycosylation_sites_literature_mining.csv
human_proteoform_glycosylation_sites_literature_mining_manually_verified.csv
human_proteoform_glycosylation_sites_oglcnac_atlas.csv
human_proteoform_glycosylation_sites_oglcnac_mcw.csv
human_proteoform_glycosylation_sites_o_gluc.csv
human_proteoform_glycosylation_sites_o_gluc_predicted.csv
human_proteoform_glycosylation_sites_pdb.csv
human_proteoform_glycosylation_sites_pdc_ccrcc.csv
human_proteoform_glycosylation_sites_platelet.csv
human_proteoform_glycosylation_sites_predicted_isoglyp.csv
human_proteoform_glycosylation_sites_rcsb_pdb.csv
human_proteoform_glycosylation_sites_tablemaker.csv
human_proteoform_glycosylation_sites_twinsuk.csv
human_proteoform_glycosylation_sites_tyr_o_linked.csv
human_proteoform_glycosylation_sites_unicarbkb.csv
human_proteoform_glycosylation_sites_unicarbkb_glycomics_study.csv
human_proteoform_glycosylation_sites_uniprotkb.csv
""".split()

import pandas as pd

# import urllib, os.path
# for filename in human_protein_mutation_files + human_proteoform_glycosylation_site_files:
#   print(f"Downloading {filename}...",end="")
#   urllib.request.urlretrieve(GLYGEN_DATA_REVIEWED+"/"+filename,"reviewed/"+filename)
#   print(" done.","(%.2f MB)"%(os.path.getsize("reviewed/"+filename)/1024**2,))

In [9]:


dfs = []
for fn in filter(lambda fn: "_uniprotkb" not in fn,human_proteoform_glycosylation_site_files):
  print(f"Download {fn}...", end="")
  df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                   usecols=["uniprotkb_canonical_ac",
                            "start_pos","start_aa",
                            "glycosylation_type"])
  print(f" done ({df.shape[0]} rows).")
  df = df[~df["uniprotkb_canonical_ac"].isna() & ~df["start_pos"].isna()]
  df['start_pos'] = df['start_pos'].astype(int)
  df["predicted"] = ("_predicted_" in fn)
  dfs.append(df)

exp_xref_keys = set(["protein_xref_pubmed","protein_xref_doi"])
for fn in filter(lambda fn: "_uniprotkb" in fn,human_proteoform_glycosylation_site_files):
  print(f"Download {fn}...", end="")
  df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                   usecols=["uniprotkb_canonical_ac",
                            "start_pos","start_aa",
                            "glycosylation_type",
                            "xref_key"])
  print(f" done ({df.shape[0]} rows).")
  df = df[~df["uniprotkb_canonical_ac"].isna() & ~df["start_pos"].isna()]
  df['start_pos'] = df['start_pos'].astype(int)
  df["predicted"] = df['xref_key'].isin(exp_xref_keys)
  df = df.drop(columns=['xref_key'])
  dfs.append(df)

glyco_sites = pd.concat(dfs,ignore_index=True)

glyco_sites_exp = glyco_sites[~glyco_sites['predicted']]
glyco_sites_exp = glyco_sites_exp.drop_duplicates()
glyco_sites_exp = glyco_sites.drop(columns=['predicted'])

glyco_sites_pred = glyco_sites[glyco_sites['predicted']]
glyco_sites_pred = glyco_sites_pred.drop_duplicates()
glyco_sites_pred = glyco_sites_pred.drop(columns=['predicted'])

glyco_sites_exp.to_csv("data/glyco_sites_exp.csv",index=False)
print(f"Wrote data/glyco_sites_exp.csv: {glyco_sites_exp.shape[0]} rows.")

glyco_sites_pred.to_csv("data/glyco_sites_pred.csv",index=False)
print(f"Wrote data/glyco_sites_pred.csv: {glyco_sites_pred.shape[0]} rows.")

del dfs, df, fn, exp_xref_keys, glyco_sites


Download human_proteoform_glycosylation_sites_carbbank.csv... done (5 rows).
Download human_proteoform_glycosylation_sites_c_man.csv... done (131 rows).
Download human_proteoform_glycosylation_sites_diabetes_glycomic.csv... done (3404 rows).
Download human_proteoform_glycosylation_sites_embl.csv... done (5571 rows).
Download human_proteoform_glycosylation_sites_glycomeatlas.csv... done (481 rows).
Download human_proteoform_glycosylation_sites_glyconnect.csv... done (30637 rows).
Download human_proteoform_glycosylation_sites_gptwiki.csv... done (1701 rows).
Download human_proteoform_glycosylation_sites_harvard.csv... done (878 rows).
Download human_proteoform_glycosylation_sites_literature.csv... done (10913 rows).
Download human_proteoform_glycosylation_sites_literature_mining.csv... done (1311 rows).
Download human_proteoform_glycosylation_sites_literature_mining_manually_verified.csv... done (166 rows).
Download human_proteoform_glycosylation_sites_oglcnac_atlas.csv... done (25661 ro

In [ ]:

dfs = []
for fn in filter(lambda fn: "_cancer" in fn,human_protein_mutation_files):
  print(f"Download {fn}...", end="")
  df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                   usecols=["uniprotkb_canonical_ac",
                            "aa_pos","ref_aa","alt_aa",
                            "do_name"])
  print(f" done ({df.shape[0]} rows).")
  df['variant_type'] = 'somatic_cancer'
  df['aa_pos'] = df['aa_pos'].astype(int)
  df = df[df["aa_pos"]>1]
  df = df[df["ref_aa"] != df["alt_aa"]]
  df['dstatus'] = (df["do_name"] != "")
  df.drop(columns=['do_name'],inplace=True)
  dfs.append(df)

for fn in filter(lambda fn: "_cancer" not in fn,human_protein_mutation_files):
  print(f"Download {fn}...")
  nrows = 0
  for df in pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                        usecols=["uniprotkb_canonical_ac",
                            "start_aa_pos","end_aa_pos",
                            "ref_aa","alt_aa"
                            "do_id","mim_id"],
                        chunksize=50000):

    df['variant_type'] = 'germline'
    df['start_aa_pos'] = df['start_aa_pos'].astype(int)
    df['end_aa_pos'] = df['end_aa_pos'].astype(int)
    df = df[df["start_aa_pos"]>1 & df['start_aa_pos'] == df['end_aa_pos'] & df["ref_aa"] != df["alt_aa"]]
    df['aa_pos'] = df['start_aa_pos']
    df['dstatus'] = (df["do_id"] != "" & df["do_mimi"] != "")
    df.drop(columns=['start_aa_pos','end_aa_pos','do_id','do_mim'],inplace=True)
    nrows += df.shape[0]
    print(f"Total rows: {nrows}")
    dfs.append(df)
  print(f" done ({nrows} rows).")

missense_variants = pd.concat(dfs,ignore_index=True)
missense_variants = missense_variants.drop_duplicates()

missense_variants.to_csv("data/missense_variants.csv",index=False)

Download human_protein_mutation_cancer_all.csv... done (4265176 rows).
Download human_protein_mutation_germline_all.csv...

In [ ]:
extract_datasets.data_dir = "data"
extract_datasets.REVIEWED_DIR = "reviewed"

extract_datasets.extract_variants()